# Bundle demand with endogenous prices BLP-style

In this notebook, I show how to estimate price sensitivity given observed choices in a market with quasi-linear prices (i.e. à la carte prices). I adopt a BLP approach:

Stage 1: combRUM recovers agent-level taste $\beta$, per-market per-item fixed effects $\delta_{tj}$, and pairwise complementarity coefficients $\lambda^{(k)}$ from a quadratic knapsack demand model with capacity constraints.

Stage 2: a 2SLS of $\widehat\delta_{tj}$ on price $p_{tj}$, instrumenting with a cost shifter $z_{tj}$.

In [1]:
import math
import os
import threading
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, replace

import numpy as np
from linearmodels.iv import IV2SLS

import combrum as cb

T, M = 15, 30
K_BETA, K_QUAD = 2, 2
n_per_market = 1000 // T
N = n_per_market * T
S = 1
CAP = 10
MASTER_BACKEND = "auto"
MAX_ITERATIONS = 80
ACTIVITY_LEVEL = "iterations"
ORACLE_WORKERS = min(12, os.cpu_count() or 1)

ALPHA_TRUE = 1.0
XI_MEAN = 0.5
NU_SCALE = 2.5
BETA_TRUE = np.array([0.6, -0.3])
LAMBDA_TRUE = np.array([0.6, 0.3])

parameters = cb.Parameters(
    {
        "beta": (-10.0, 10.0, K_BETA),
        "delta": (-10.0, 10.0, T * M),
        "lambda": (0.0, 5.0, K_QUAD),
    }
)

print(f"N={N}  T={T}  M={M}  S={S}  K={parameters.K}")

N=990  T=15  M=30  S=1  K=454


## Synthetic data

$$p_{tj} = 0.7\, z_{tj} + \xi_{tj} + 0.3\, \zeta^p_{tj}$$
$$\delta_{tj} = -\alpha\, p_{tj} + \xi_{tj} + 0.5\, \zeta^\delta_{tj}$$

$\xi_{tj}$ has nonzero mean and enters both $p_{tj}$ and $\delta_{tj}$. $z_{tj}$ enters only $p_{tj}$. The $K_{\text{quad}}$ matrices $Q^{(k)} \in \{0,1\}^{M \times M}$ are symmetric and sparse, with $Q^{(k)}_{jj'} = 1$ marking an interaction between items $j$ and $j'$ at layer $k$.

In [2]:
@dataclass(frozen=True)
class BLPDesign:
    X: np.ndarray
    market: np.ndarray
    weights: np.ndarray
    capacity: float
    Q: np.ndarray
    instruments: np.ndarray
    prices: np.ndarray
    delta_true: np.ndarray
    theta_true: np.ndarray
    dgp_shocks: np.ndarray
    est_shocks: np.ndarray
    observed: np.ndarray

    @property
    def N(self):
        return self.X.shape[0]

    @property
    def M(self):
        return self.X.shape[1]


def build_design():
    rng = np.random.default_rng(0)
    market = np.repeat(np.arange(T), n_per_market)
    X = rng.normal(size=(N, M, K_BETA))
    weights = np.ones(M)

    Q = np.zeros((K_QUAD, M, M))
    for k in range(K_QUAD):
        upper = np.triu(rng.random((M, M)) < 0.10, 1)
        Q[k] = upper + upper.T

    xi = XI_MEAN + rng.normal(size=(T, M))
    instruments = rng.normal(size=(T, M))
    prices = 0.7 * instruments + xi + 0.3 * rng.normal(size=(T, M))
    delta_true = -ALPHA_TRUE * prices + xi + 0.5 * rng.normal(size=(T, M))
    theta_true = parameters.pack(
        {"beta": BETA_TRUE, "delta": delta_true.ravel(), "lambda": LAMBDA_TRUE}
    )
    return BLPDesign(
        X=X,
        market=market,
        weights=weights,
        capacity=CAP,
        Q=Q,
        instruments=instruments,
        prices=prices,
        delta_true=delta_true,
        theta_true=theta_true,
        dgp_shocks=NU_SCALE * rng.normal(size=(N, M)),
        est_shocks=NU_SCALE * rng.normal(size=(N, S, M)),
        observed=np.zeros((N, M), dtype=bool),
    )


design = build_design()

## Batched feature map

The feature map is where we define the statistics of a bundle: item characteristics, market-item fixed effects, and pairwise interactions. combRUM calls `features_batch` both for new bundles found during row generation and for the observed bundles in the data.

In [3]:

class BLPFeatures(cb.FeatureMap):
    def __init__(self, design):
        self.design = design

    def features_batch(
        self,
        ids,
        bundles,
        weights=None,
        aggregate=False,
    ):
        obs = ids % self.design.N
        sim = ids // self.design.N
        q_start = K_BETA + T * M
        feature_dim = q_start + K_QUAD
        delta_cols = self.design.market[obs, None] * M + np.arange(M)

        if aggregate:
            phi = np.zeros(feature_dim)
            phi[:K_BETA] = np.einsum(
                "n,nm,nmk->k", weights, bundles, self.design.X[obs], optimize=True
            )
            phi[K_BETA:q_start] = np.bincount(
                delta_cols.ravel(),
                weights=(weights[:, None] * bundles).ravel(),
                minlength=T * M,
            )
            phi[q_start:] = np.einsum(
                "n,ni,kij,nj->k", weights, bundles, self.design.Q, bundles, optimize=True
            )
            eps = np.einsum(
                "n,nm,nm->", weights, self.design.est_shocks[obs, sim], bundles, optimize=True
            )
            return phi, eps

        Phi = np.zeros((ids.size, feature_dim))
        Phi[:, :K_BETA] = np.einsum(
            "nm,nmk->nk", bundles, self.design.X[obs], optimize=True
        )
        Phi[np.arange(ids.size)[:, None], K_BETA + delta_cols] = bundles
        Phi[:, q_start:] = np.einsum(
            "ni,kij,nj->nk", bundles, self.design.Q, bundles, optimize=True
        )
        eps = np.einsum("nm,nm->n", self.design.est_shocks[obs, sim], bundles, optimize=True)
        return Phi, eps


## The demand oracle

Each agent in market $t$ solves

$$\max_{d \in \{0,1\}^M} \,\sum_j d_j \big( X_{ij}^\top \beta + \delta_{t,j} + \nu_{ij} \big) + \sum_k \lambda^{(k)}\, d^\top Q^{(k)} d \quad \text{s.t.} \quad \sum_j w_j d_j \le c_i.$$

`BLPDemandOracle` is the demand oracle for this example. Given the current parameter values, it solves each agent’s quadratic knapsack problem. The notebook uses Gurobi when available, and otherwise falls back to HiGHS.

In [4]:

def choose_demand_backend():
    try:
        import gurobipy

        env = gurobipy.Env(empty=True)
        env.setParam("OutputFlag", 0)
        env.start()
        env.dispose()
        return "gurobi"
    except Exception:
        return "highs"


In [ ]:
class GurobiDemand:
    def __init__(self, design):
        import gurobipy as gp

        self.gp = gp
        self.env = gp.Env(empty=True)
        self.env.setParam("OutputFlag", 0)
        self.env.start()
        self.model = gp.Model(env=self.env)
        self.model.Params.OutputFlag = 0
        self.x = self.model.addMVar(design.M, vtype=gp.GRB.BINARY)
        self.model.addConstr(design.weights @ self.x <= design.capacity)
        self.model.update()

    def apply_solver_settings(self, settings):
        if settings.time_limit_seconds is not None:
            self.model.Params.TimeLimit = settings.time_limit_seconds
        if settings.mip_focus is not None:
            self.model.Params.MIPFocus = settings.mip_focus

    def solve(self, linear, quadratic):
        self.model.setMObjective(
            Q=quadratic, c=linear, constant=0.0, sense=self.gp.GRB.MAXIMIZE
        )
        self.model.optimize()
        if self.model.Status != self.gp.GRB.OPTIMAL and self.model.SolCount == 0:
            raise RuntimeError("Gurobi returned no feasible bundle")
        bundle = self.x.X > 0.5
        value = self.model.ObjVal
        raw_gap = self.model.MIPGap
        if self.model.Status == self.gp.GRB.OPTIMAL and math.isfinite(raw_gap) and raw_gap <= 0.0:
            return cb.Demand.exact(bundle, value)
        return cb.Demand.uncertified(bundle, value, gap=raw_gap)

    def close(self):
        self.model.dispose()
        self.env.dispose()


In [6]:

class HighsDemand:
    def __init__(self, design):
        import highspy

        self.highspy = highspy
        upper = np.triu(np.any(design.Q != 0.0, axis=0), 1)
        self.n_items = design.M
        self.edges = np.column_stack(np.nonzero(upper))
        self.edge_rows = self.edges[:, 0]
        self.edge_cols = self.edges[:, 1]
        self.costs = np.empty(self.n_items + self.edges.shape[0])
        self.cost_indices = np.arange(self.costs.size, dtype=np.int32)
        self.model = self._build_model(design)

    def _build_model(self, design):
        hp = self.highspy
        n_edges = self.edges.shape[0]
        n_vars = design.M + n_edges
        model = hp.Highs()
        model.setOptionValue("output_flag", False)
        model.setOptionValue("threads", 1)
        model.setOptionValue("parallel", "off")
        model.setOptionValue("presolve", "off")
        model.setOptionValue("mip_rel_gap", 0.0)
        model.addVars(n_vars, np.zeros(n_vars), np.ones(n_vars))
        model.changeColsIntegrality(
            design.M,
            np.arange(design.M, dtype=np.int32),
            np.full(design.M, hp.HighsVarType.kInteger.value, dtype=np.uint8),
        )
        model.addRow(
            -hp.kHighsInf,
            design.capacity,
            design.M,
            np.arange(design.M, dtype=np.int32),
            design.weights,
        )
        for edge_id, (j, k) in enumerate(self.edges):
            y = design.M + edge_id
            model.addRow(-hp.kHighsInf, 0.0, 2, np.array([y, j], dtype=np.int32), np.array([1.0, -1.0]))
            model.addRow(-hp.kHighsInf, 0.0, 2, np.array([y, k], dtype=np.int32), np.array([1.0, -1.0]))
            model.addRow(-hp.kHighsInf, 1.0, 3, np.array([j, k, y], dtype=np.int32), np.array([1.0, 1.0, -1.0]))
        model.setMaximize()
        return model

    def apply_solver_settings(self, settings):
        if settings.time_limit_seconds is not None:
            self.model.setOptionValue("time_limit", settings.time_limit_seconds)

    def solve(self, linear, quadratic):
        self.costs[: self.n_items] = linear
        self.costs[self.n_items :] = 2.0 * quadratic[self.edge_rows, self.edge_cols]
        self.model.changeColsCost(self.costs.size, self.cost_indices, self.costs)
        status = self.model.run()
        model_status = self.model.getModelStatus()
        solution = self.model.getSolution()
        ok = {
            self.highspy.HighsModelStatus.kOptimal,
            self.highspy.HighsModelStatus.kTimeLimit,
            self.highspy.HighsModelStatus.kSolutionLimit,
        }
        run_ok = {self.highspy.HighsStatus.kOk, self.highspy.HighsStatus.kWarning}
        if status not in run_ok or model_status not in ok or not solution.value_valid:
            raise RuntimeError(f"HiGHS returned no feasible bundle: {model_status}")
        bundle = np.asarray(solution.col_value[: self.n_items]) > 0.5
        value = self.model.getObjectiveValue()
        raw_gap = self.model.getInfo().mip_gap
        if model_status == self.highspy.HighsModelStatus.kOptimal and math.isfinite(raw_gap) and raw_gap <= 0.0:
            return cb.Demand.exact(bundle, value)
        return cb.Demand.uncertified(bundle, value, gap=raw_gap)

    def close(self):
        self.model.clear()


In [7]:

class BLPDemandOracle(cb.Oracle):
    def __init__(self, design, shocks):
        self.design = design
        self.shocks = shocks
        self.settings = cb.SolverSettings()
        self.solvers = {}
        self.solver_lock = threading.Lock()

    def _solver(self, agent_id):
        with self.solver_lock:
            solver = self.solvers.get(agent_id)
            if solver is not None:
                return solver
            settings = self.settings
        if DEMAND_BACKEND == "gurobi":
            solver = GurobiDemand(self.design)
        else:
            solver = HighsDemand(self.design)
        solver.apply_solver_settings(settings)
        with self.solver_lock:
            self.solvers[agent_id] = solver
        return solver

    def apply_solver_settings(self, settings):
        updated = cb.SolverSettings(
            time_limit_seconds=(
                self.settings.time_limit_seconds
                if settings.time_limit_seconds is None
                else settings.time_limit_seconds
            ),
            mip_focus=self.settings.mip_focus if settings.mip_focus is None else settings.mip_focus,
        )
        if updated == self.settings:
            return
        self.settings = updated
        with self.solver_lock:
            solvers = tuple(self.solvers.values())
        for solver in solvers:
            solver.apply_solver_settings(updated)

    def price_batch(self, theta, local_ids):
        values = parameters.unpack(theta)
        beta = values["beta"]
        delta = values["delta"].reshape(T, M)
        quadratic = np.tensordot(values["lambda"], self.design.Q, axes=([0], [0]))

        obs = local_ids % self.design.N
        sim = local_ids // self.design.N
        unique_obs, obs_pos = np.unique(obs, return_inverse=True)
        base_linear = (
            np.einsum("umk,k->um", self.design.X[unique_obs], beta, optimize=True)
            + delta[self.design.market[unique_obs]]
        )

        def solve_one(row):
            agent_id = local_ids[row]
            linear = base_linear[obs_pos[row]] + self.shocks[obs[row], sim[row]]
            demand = self._solver(agent_id).solve(linear, quadratic)
            return agent_id, demand

        with ThreadPoolExecutor(max_workers=ORACLE_WORKERS) as pool:
            return dict(pool.map(solve_one, range(len(local_ids))))

    def teardown(self):
        with self.solver_lock:
            solvers = tuple(self.solvers.values())
            self.solvers.clear()
        for solver in solvers:
            solver.close()


DEMAND_BACKEND = choose_demand_backend()
print(f"demand backend: {DEMAND_BACKEND}")


demand backend: gurobi


## Simulate observed bundles at $\theta_{\text{true}}$

In [8]:

def simulate_observed(design):
    demand = BLPDemandOracle(
        design,
        design.dgp_shocks.reshape(N, 1, M),
    )
    ids = np.arange(N)
    priced = demand.price_batch(design.theta_true, ids)
    demand.teardown()
    observed = np.vstack([priced[i].bundle for i in ids])
    return replace(design, observed=observed)


design = simulate_observed(design)
item_counts = np.zeros((T, M), dtype=np.int64)
np.add.at(item_counts, design.market, design.observed)
if item_counts.min() == 0:
    t, j = np.argwhere(item_counts == 0)[0]
    raise ValueError(f"DGP left item {j} unchosen in market {t}")

print(f"mean bundle size: {design.observed.sum(axis=1).mean():.2f} / cap {CAP}")


mean bundle size: 10.00 / cap 10


## Stage 1: combRUM row generation

The observed data is $\{\hat d_i\}_{i=1}^N \subset \{0,1\}^M$, the realized choices of each agent. We also see $X_i$, $w$, $c_i$, $t(i)$, and the $Q^{(k)}$.

The parameters $\theta = (\beta, \delta, \lambda)$ are estimated by the LP

$$
\begin{aligned}
\min_{\theta,\, u \ge 0} \,\, & \sum_{i,s} u_{is} \,-\, \sum_{i,s} \phi(\hat d_i, x_i)^\top \theta \\
\text{s.t.} \,\, & u_{is} \,\ge\, \phi(d, x_i)^\top \theta + \nu_{is}^\top d \qquad \forall\, i, s,\, d \in \mathcal{C}_i,
\end{aligned}
$$

where $\mathcal{C}_i = \{d \in \{0,1\}^M : w^\top d \le c_i\}$ and $\phi(d, x_i)$ has three blocks: $X_i^\top d$ for $\beta$, $e_{t(i)} \otimes d$ for $\delta$, and $(d^\top Q^{(k)} d)_k$ for $\lambda$.

In [9]:

features = BLPFeatures(design)
demand = BLPDemandOracle(
    design,
    design.est_shocks,
)
model = cb.Model(demand, parameters, features=features, formulation=cb.NSlack)
data = cb.Data(
    observed_bundles=design.observed,
    shocks=design.est_shocks,
    observables=np.arange(N),
)

time_limit_callback = cb.point_timeout_callback(
    cb.Schedule((cb.Phase(timeout=1.0, iters=30), cb.Phase(timeout=20.0)))
)
fit = cb.estimate(
    model,
    data,
    master_backend=MASTER_BACKEND,
    tolerance=1e-3,
    max_iterations=MAX_ITERATIONS,
    cut_policy=cb.AddAll(),
    iteration_callback=time_limit_callback,
    activity=cb.ActivityConfig(
        label="blp",
        level=ACTIVITY_LEVEL,
        stdout=True,
    ),
)


[blp] row generation: N=990 S=1 K=454 agents=990 ranks=1 tol=1.000e-03 max_iter=80 cuts=AddAll


[blp] iter         gap        dgap         obj        dobj  +cuts   cuts  priced  inexact   price  master      dt    total


[blp]    0   5.214e+02           -  -7.780e+04           -    990    990     990       36   6.41s   0.01s   6.42s    6.42s


[blp]    1   4.506e+02  -7.080e+01  -1.013e+04   6.768e+04    990   1980     990        4   1.98s   0.05s   2.04s    8.47s


[blp]    2   1.332e+02  -3.175e+02   2.741e+03   1.287e+04    990   2970     990        0   0.20s   0.15s   0.35s    8.83s


...

[blp]   26   2.132e-14   0.000e+00   2.122e+04   0.000e+00      0  14806     990       15   2.54s   0.00s   2.54s   77.43s


[blp]   27   2.132e-14   0.000e+00   2.122e+04   0.000e+00      0  14806     990       15   2.55s   0.00s   2.55s   79.98s


[blp]   28   2.132e-14   0.000e+00   2.122e+04   0.000e+00      0  14806     990       15   2.49s   0.00s   2.49s   82.46s


[blp]   29   2.132e-14   0.000e+00   2.122e+04   0.000e+00      0  14806     990       15   2.51s   0.00s   2.51s   84.97s


[blp] done converged=yes reason=converged iters=30 gap=2.132e-14 cuts=14806 obj=2.122e+04 wall=84.97s


In [10]:
named = fit.theta_named()
beta_hat = named["beta"]
delta_hat = named["delta"].reshape(T, M)
lambda_hat = named["lambda"]
print(f"iterations: {fit.metadata['iterations']}   converged: {fit.metadata['converged']}   wall: {fit.runtime_seconds:.1f}s")
print(f"beta_true:   {BETA_TRUE}")
print(f"beta_hat:    {beta_hat}")
print(f"lambda_true: {LAMBDA_TRUE}")
print(f"lambda_hat:  {lambda_hat}")
print(f"delta correlation: {np.corrcoef(delta_hat.ravel(), design.delta_true.ravel())[0, 1]:.3f}")

iterations: 30   converged: True   wall: 85.1s
beta_true:   [ 0.6 -0.3]
beta_hat:    [ 0.59025796 -0.34242406]
lambda_true: [0.6 0.3]
lambda_hat:  [0.63888854 0.31861788]
delta correlation: 0.792


## Stage 2: 2SLS

Stack the market-item observations $(t, j)$ and regress $\widehat\delta_{tj}$ on price, instrumenting with the excluded cost shifter $z_{tj}$. For comparison, we also run the same second stage using the true $\delta_{tj}$ from the simulation.

In [11]:
def price_sensitivity_alpha(delta_tj):
    delta = delta_tj.ravel()
    price = design.prices.ravel()
    instrument = design.instruments.ravel()
    constant = np.ones(delta.size)
    ols = IV2SLS(delta, np.column_stack([constant, price]), None, None).fit()
    tsls = IV2SLS(delta, constant, price, instrument).fit()
    return -ols.params.iloc[-1], -tsls.params.iloc[-1]


ols_alpha, tsls_alpha = price_sensitivity_alpha(delta_hat)
true_ols_alpha, true_tsls_alpha = price_sensitivity_alpha(
    design.delta_true
)

print("target alpha:", round(ALPHA_TRUE, 6))
print("ols alpha (estimated delta):", round(ols_alpha, 6))
print("2sls alpha (estimated delta):", round(tsls_alpha, 6))
print("ols alpha (true delta):", round(true_ols_alpha, 6))
print("2sls alpha (true delta):", round(true_tsls_alpha, 6))

target alpha: 1.0
ols alpha (estimated delta): 0.382685
2sls alpha (estimated delta): 0.980509
ols alpha (true delta): 0.351201
2sls alpha (true delta): 0.93084
